# 14장. 모델 해석: 특성 중요도와 부분 의존도

트리 수백 개를 더한 모델이 어떤 특성을 어떻게 쓰는지 알아내는 방법과, 그 방법이 틀리게 답하는 경우.

**준비:** CSV 다섯 개(서울 아파트 실거래가 넷, 경기도 중국집 하나)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 5분쯤 걸린다.

## 14.1 불순도 감소로 구한 특성 중요도

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def brier(actual, prob):
    return float(((prob - actual) ** 2).mean())

NAMES = ["업력", "위도", "경도", "지번폐업", "주변", "주변폐업", "시군 번호", "난수"]

def eight(df, seed):
    """12장의 특성 7개에, 가게와 아무 관계가 없는 0과 1 사이의 난수를 여덟 번째 특성으로 추가한다."""
    return np.column_stack([df["업력"], df["위도"].fillna(0), df["경도"].fillna(0), df["지번폐업"], df["주변"].fillna(-1),
                            df["주변폐업"].fillna(-1), pd.Index(SIGUN).get_indexer(df["시군"]),
                            np.random.default_rng(seed).random(len(df))])

year = train["기준일"].dt.year.to_numpy()
fit_on, tune_on = year < 2020, year == 2020
R, Rt = eight(train, 1), eight(test, 2)
deep = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(R[fit_on], ytr[fit_on])                           # 기본 설정: 끝까지 키운 트리
wide = RandomForestClassifier(300, min_samples_leaf=200, n_jobs=-1, random_state=0).fit(R[fit_on], ytr[fit_on])     # 12장에서 고른 설정
own = pd.DataFrame({"기본 설정": deep.feature_importances_, "잎에 200줄": wide.feature_importances_}, index=NAMES).round(3)
print(own.to_string())
for name, m in [("기본 설정", deep), ("잎에 200줄", wide)]:
    print(name, round(auc(ytr[fit_on], m.predict_proba(R[fit_on])[:, 1]), 3), round(auc(ytr[tune_on], m.predict_proba(R[tune_on])[:, 1]), 3),
          round(float(np.mean([e.get_n_leaves() for e in m.estimators_]))))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 14-1
order = own["기본 설정"].sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), sharey=True)
for ax, col in zip(axes, own.columns):
    vals = own.loc[order, col]
    ax.barh(order, vals, color=[BRICK if n == "난수" else NAVY for n in order], height=0.62)
    for yy, val in enumerate(vals):
        ax.text(val + 0.004, yy, f"{val:.3f}", fontsize=7.5, va="center", color=BRICK if order[yy] == "난수" else NAVY)
    ax.set_xlim(0, 0.26)
    ax.set_xlabel("불순도 감소로 구한 중요도(합이 1)", fontsize=8)
    frame(ax, col, grid="x")
fig.tight_layout()
fig.savefig("fig_14_1.png", dpi=300)
plt.show()

In [ ]:
def impurity_gain(tree):
    """트리 하나의 분할마다 (특성 번호, 그 노드의 줄 수, 가중 불순도 감소)를 구한다."""
    t = tree.tree_
    inner = t.children_left >= 0                                                # 잎 노드가 아닌 노드
    n, imp, left, right = t.weighted_n_node_samples, t.impurity, t.children_left, t.children_right
    gain = (n[inner] * imp[inner] - n[left[inner]] * imp[left[inner]] - n[right[inner]] * imp[right[inner]]) / n[0]
    return t.feature[inner], n[inner], gain

first = deep.estimators_[0]
feat, size, gain = impurity_gain(first)
mine = np.bincount(feat, weights=gain, minlength=8)
print(len(feat), np.abs(mine / mine.sum() - first.feature_importances_).max().round(12))

BINS = [0, 20, 100, 1000, 1e9]
LABELS = ["20줄 미만", "20~99줄", "100~999줄", "1,000줄 이상"]
share = np.zeros((8, 4))
count = np.zeros((8, 4))
for e in deep.estimators_:
    feat, size, gain = impurity_gain(e)
    where_ = np.digitize(size, BINS) - 1
    np.add.at(share, (feat, where_), gain)
    np.add.at(count, (feat, where_), 1)
share = share / share.sum()
by_size = pd.DataFrame(share, index=NAMES, columns=LABELS).round(3)
print(by_size.to_string())
print((count.sum(axis=1) / 300).round(0).astype(int).tolist())
print(train["지번폐업"].nunique(), round(float((train["지번폐업"] <= 3).mean()) * 100, 1))            # 지번폐업의 값의 종류, 0~3인 줄의 비율(%)

In [ ]:
# 그림 14-2
order = own["기본 설정"].sort_values().index
fig, ax = plt.subplots(figsize=(7.2, 3.9))
leftv = np.zeros(8)
for col, color in zip(LABELS[::-1], [NAVY, TEAL, "#d9a58f", BRICK]):
    vals = by_size.loc[order, col].to_numpy()
    ax.barh(order, vals, left=leftv, color=color, height=0.62, label=f"노드의 줄 수 {col}", edgecolor="white", lw=0.5)
    leftv += vals
ax.set_xlabel("불순도 감소의 합에서 차지하는 비율(트리 300개 전체)", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_14_2.png", dpi=300)
plt.show()

## 14.2 순열 중요도

In [ ]:
def drop_when_shuffled(predict, M, actual, metric, repeats=10, seed=0):
    """특성을 하나씩 줄 사이에서 무작위로 섞어서 다시 예측하고, 지표가 섞기 전보다 얼마나 줄어드는지 구한다. 특성마다 (평균, 표준편차)."""
    rng = np.random.default_rng(seed)
    before = metric(actual, predict(M))
    out = []
    for j in range(M.shape[1]):
        falls = []
        for _ in range(repeats):
            mixed = M.copy()
            mixed[:, j] = rng.permutation(mixed[:, j])                           # 이 특성만 섞는다
            falls.append(before - metric(actual, predict(mixed)))
        out.append((np.mean(falls), np.std(falls)))
    return np.array(out)

prob_of = lambda forest: (lambda M: forest.predict_proba(M)[:, 1])
on_fit = drop_when_shuffled(prob_of(deep), R[fit_on], ytr[fit_on], auc, repeats=3)
on_new = drop_when_shuffled(prob_of(deep), R[tune_on], ytr[tune_on], auc)
on_new_wide = drop_when_shuffled(prob_of(wide), R[tune_on], ytr[tune_on], auc)
perm = pd.DataFrame({"기본/훈련": on_fit[:, 0], "기본/검증": on_new[:, 0], "편차": on_new[:, 1],
                     "200줄/검증": on_new_wide[:, 0], "편차 ": on_new_wide[:, 1]}, index=NAMES).round(4)
print(perm.to_string())

In [ ]:
# 그림 14-4
order = perm["기본/훈련"].sort_values().index
ypos = np.arange(8)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.7), sharey=True)
axes[0].barh(ypos, perm.loc[order, "기본/훈련"], color=[BRICK if n == "난수" else NAVY for n in order], height=0.62)
frame(axes[0], "훈련 자료에서 섞었을 때", grid="x")
axes[1].barh(ypos, perm.loc[order, "기본/검증"], xerr=2 * perm.loc[order, "편차"], color=[BRICK if n == "난수" else NAVY for n in order],
             height=0.62, error_kw={"ecolor": GRAY, "lw": 1.2, "capsize": 2})
frame(axes[1], "2020년 검증 자료에서 섞었을 때", grid="x")
axes[0].set_yticks(ypos, order, fontsize=8)
for ax in axes:
    ax.set_xlabel("AUC가 줄어든 양", fontsize=8)
axes[1].axvline(0, color=GRAY, lw=0.8)
fig.tight_layout()
fig.savefig("fig_14_4.png", dpi=300)
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

theirs = permutation_importance(wide, R[tune_on], ytr[tune_on], scoring="roc_auc", n_repeats=10, random_state=0)
print({n: round(float(m), 4) for n, m in zip(NAMES, theirs.importances_mean)})

In [ ]:
rows = []
whole = auc(ytr[tune_on], wide.predict_proba(R[tune_on])[:, 1])
for j, name in enumerate(NAMES):
    keep = [k for k in range(8) if k != j]
    less = RandomForestClassifier(300, min_samples_leaf=200, n_jobs=-1, random_state=0).fit(R[fit_on][:, keep], ytr[fit_on])
    rows.append((name, perm.loc[name, "200줄/검증"], round(whole - auc(ytr[tune_on], less.predict_proba(R[tune_on][:, keep])[:, 1]), 4)))
where3 = [1, 2, 6]                                                               # 위도, 경도, 시군 번호
rng = np.random.default_rng(0)
falls = []
for _ in range(10):
    mixed = R[tune_on].copy()
    mixed[:, where3] = mixed[rng.permutation(len(mixed))][:, where3]             # 세 특성을 같은 순서로 함께 섞는다
    falls.append(whole - auc(ytr[tune_on], wide.predict_proba(mixed)[:, 1]))
keep = [k for k in range(8) if k not in where3]
less = RandomForestClassifier(300, min_samples_leaf=200, n_jobs=-1, random_state=0).fit(R[fit_on][:, keep], ytr[fit_on])
rows.append(("위도 + 경도 + 시군 번호", round(float(np.mean(falls)), 4), round(whole - auc(ytr[tune_on], less.predict_proba(R[tune_on][:, keep])[:, 1]), 4)))
print(round(whole, 3))
print(pd.DataFrame(rows, columns=["특성", "섞었을 때", "빼고 다시 추정"]).to_string(index=False))

In [ ]:
import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
a23, a24, a25 = v[v["연"] == 2023], v[v["연"] == 2024], v[v["연"] == 2025]
both = v[v["연"] <= 2024]

def off(actual_log, predicted_log):
    """오차율의 중앙값(%). 2장의 off와 같은 지표를 로그 가격에서 계산한다."""
    return round(float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100), 1)

def log_rmse(actual_log, predicted_log):
    return round(float(np.sqrt(((predicted_log - actual_log) ** 2).mean())), 4)

def table_of(df, ref, month=False):
    """면적, 층, 건축년도는 숫자 그대로 쓴다. 동과 단지는 ref에서 가나다순으로 매긴 번호로 바꾼다. 목록에 없는 이름은 -1이 된다."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    cols = [df["면적"], df["층"], df["건축년도"], dongs.get_indexer(df["동이름"]), complexes.get_indexer(df["단지"])]
    return np.column_stack(cols + ([df["달"]] if month else []))

from scipy import sparse
from sklearn.linear_model import Ridge

def design(df, ref):
    """10장의 특성 행렬에 계약한 달을 수치 특성 하나로 추가했다. 수치 특성 8개에 동과 단지의 가변수."""
    dongs, complexes = pd.Index(sorted(ref["동이름"].unique())), pd.Index(sorted(ref["단지"].unique()))
    num = np.column_stack([np.log(df["면적"]), df["층"], *((np.clip(df["연식"], 0, 59) // 10 == k).astype(float) for k in range(1, 6)), df["달"]])
    blocks = [sparse.csr_matrix(num)]
    for col, cats in [("동이름", dongs), ("단지", complexes)]:
        where_ = cats.get_indexer(df[col])
        known = where_ >= 0
        blocks.append(sparse.csr_matrix((np.ones(known.sum()), (np.where(known)[0], where_[known])), shape=(len(df), len(cats))))
    return sparse.hstack(blocks).tocsr()

from sklearn.ensemble import HistGradientBoostingRegressor

both, a25 = both.reset_index(drop=True), a25.reset_index(drop=True)
pb, p25 = np.log(both["억"].to_numpy()), np.log(a25["억"].to_numpy())
line = Ridge(alpha=0.01, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(design(both, both), pb)
mend = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=1000, early_stopping=False, random_state=0).fit(
    table_of(both, both, month=True), pb - line.predict(design(both, both)))
COLS = ["면적", "층", "건축년도", "동이름", "단지", "달"]

def model(df):
    """13.4절의 모델. 거래 표를 받아 로그 가격을 예측한다. 릿지의 예측에 잔차를 맡은 부스팅의 예측을 더한다."""
    d = df.assign(연식=df["연"] - df["건축년도"])                               # 건축년도를 바꾸면 연식도 따라 바뀌게 한다
    return line.predict(design(d, both)) + mend.predict(table_of(d, both, month=True))

before = log_rmse(p25, model(a25))
print(off(p25, model(a25)), before)

In [ ]:
def rise_when_shuffled(cols, repeats=3, seed=0):
    """2025년 표에서 cols의 특성들을 같은 순서로 함께 섞는다. 로그 RMSE가 얼마나 커지는지 돌려준다."""
    rng = np.random.default_rng(seed)
    rises = []
    for _ in range(repeats):
        order = rng.permutation(len(a25))
        mixed = a25.copy()
        for col in cols:
            mixed[col] = a25[col].to_numpy()[order]
        rises.append(log_rmse(p25, model(mixed)) - before)
    return round(float(np.mean(rises)), 4)

apt_perm = {col: rise_when_shuffled([col]) for col in COLS}
apt_perm["동이름 + 단지"] = rise_when_shuffled(["동이름", "단지"])
apt_perm["동이름 + 단지 + 건축년도"] = rise_when_shuffled(["동이름", "단지", "건축년도"])
print(apt_perm)
per_complex = both.groupby("단지")[["건축년도", "동이름"]].nunique()
print(len(per_complex), int((per_complex["건축년도"] > 1).sum()), int((per_complex["동이름"] > 1).sum()))

In [ ]:
# 그림 14-5
names = list(apt_perm)[::-1]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
vals = [apt_perm[n] for n in names]
ax.barh(names, vals, color=[TEAL if "+" in n else NAVY for n in names], height=0.6)
for yy, val in enumerate(vals):
    ax.text(val + 0.008, yy, f"+{val:.4f}", fontsize=7.5, va="center", color=TEAL if "+" in names[yy] else NAVY)
ax.set_xlim(0, 0.68)
ax.set_xlabel(f"섞었을 때 2025년 로그 RMSE가 커진 양(섞기 전 {before:.4f})", fontsize=8)
frame(ax, grid="x")
ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig("fig_14_5.png", dpi=300)
plt.show()

## 14.3 부분 의존도와 개별 조건부 기대 곡선

In [ ]:
bands = pd.cut(a25["층"], [0, 1, 2, 3, 5, 10, 15, 20, 25, 70], labels=["1층", "2층", "3층", "4~5층", "6~10층", "11~15층", "16~20층", "21~25층", "26층~"])
raw_floor = a25.groupby(bands, observed=True)["억"].agg(["median", "size"])
print({k: (round(float(m), 2), int(s)) for k, (m, s) in raw_floor.iterrows()})

some = a25.sample(300, random_state=0)                                           # 2025년의 거래 300건
FLOORS = [1, 2, 3, 5, 10, 15, 20, 25, 30]
asked = np.array([model(some.assign(층=f)) for f in FLOORS])                     # 층만 바꿔서 300건을 다시 예측한다. 행이 층, 열이 거래
relative = (np.exp(asked - asked[4]) - 1) * 100                                  # 같은 집이 10층일 때에 비해 몇 %인가
print({f: round(float(m), 1) for f, m in zip(FLOORS, relative.mean(axis=1))})
print(np.quantile(relative[0], [0.1, 0.5, 0.9]).round(1).tolist(), np.quantile(relative[-1], [0.1, 0.5, 0.9]).round(1).tolist())
top = both.groupby("단지")["층"].max()                                           # 단지마다 거래된 가장 높은 층
tall = some["단지"].map(top).fillna(some["층"]).to_numpy() >= 25
print(int(tall.sum()), round(float(relative[-1][tall].mean()), 1), round(float(relative[-1][~tall].mean()), 1), round(float((both["층"] >= 26).mean()) * 100, 1))

In [ ]:
# 그림 14-6
fig, (ax, low) = plt.subplots(2, 1, figsize=(7.2, 5.4), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for j in range(100):
    ax.plot(FLOORS, relative[:, j], color=GRAY, lw=0.5, alpha=0.6)
ax.plot([], [], color=GRAY, lw=0.8, label="거래 한 건(100건만 그렸다)")
ax.plot(FLOORS, relative[:, tall].mean(axis=1), color=TEAL, lw=1.6, ls="--", label=f"25층 이상이 있는 단지의 거래 {int(tall.sum())}건의 평균")
ax.plot(FLOORS, relative.mean(axis=1), color=BRICK, lw=2.4, label="300건의 평균(부분 의존도)")
ax.axhline(0, color=NAVY, lw=0.6)
ax.set_ylabel("같은 집이 10층일 때에 비한 예측 가격(%)", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
frame(ax)
low.hist(both["층"].clip(upper=31), bins=np.arange(0.5, 32.5, 1), color=NAVY, alpha=0.75)
low.set_ylabel("훈련 자료의\n거래 수", fontsize=8)
low.set_xlabel("층(31은 31층 이상)", fontsize=8)
frame(low)
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout()
fig.savefig("fig_14_6.png", dpi=300)
plt.show()

In [ ]:
YEARS_BUILT = [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
asked_b = np.array([model(some.assign(건축년도=b)) for b in YEARS_BUILT])
rel_b = (np.exp(asked_b - asked_b[4]) - 1) * 100                                 # 같은 집이 2000년에 지은 집일 때에 비해 몇 %인가
print({b: round(float(m), 1) for b, m in zip(YEARS_BUILT, rel_b.mean(axis=1))})
print({int(k): round(float(m), 1) for k, m in a25.groupby((a25["건축년도"] // 10) * 10)["억"].median().items()})
lin_only = np.array([line.predict(design(some.assign(연식=some["연"] - b), both)) for b in YEARS_BUILT])
print({b: round(float(m), 1) for b, m in zip(YEARS_BUILT, ((np.exp(lin_only - lin_only[4]) - 1) * 100).mean(axis=1))})
print(np.quantile(rel_b[0], [0.1, 0.9]).round(1).tolist(), np.quantile(rel_b[-1], [0.1, 0.9]).round(1).tolist())

In [ ]:
# 그림 14-7
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for j in range(100):
    ax.plot(YEARS_BUILT, rel_b[:, j], color=GRAY, lw=0.5, alpha=0.6)
ax.plot([], [], color=GRAY, lw=0.8, label="거래 한 건(100건만 그렸다)")
ax.plot(YEARS_BUILT, ((np.exp(lin_only - lin_only[4]) - 1) * 100).mean(axis=1), color=NAVY, lw=1.6, ls="--", label="릿지 부분만의 평균")
ax.plot(YEARS_BUILT, rel_b.mean(axis=1), color=BRICK, lw=2.4, label="300건의 평균(부분 의존도)")
ax.axhline(0, color=NAVY, lw=0.6)
ax.set_xlabel("건축년도를 이 값으로 바꿨을 때", fontsize=8)
ax.set_ylabel("같은 집이 2000년에 지은 집일 때에 비한 예측 가격(%)", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="upper center", ncol=3)
ax.set_ylim(rel_b[:, :100].min() - 2, rel_b[:, :100].max() + 9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_14_7.png", dpi=300)
plt.show()

## 14.4 한 건의 예측을 특성별로 나누기, 해석으로 찾은 누수

In [ ]:
import itertools, math

crowd = both.sample(200, random_state=0).reset_index(drop=True)                  # 비교 기준: 훈련 자료의 거래 200건
one = a25.iloc[[0]]                                                              # 설명할 거래 하나
print(one[["동이름", "면적", "층", "건축년도", "달", "억"]].to_string(index=False))

def worth(known):
    """known에 있는 특성만 이 거래의 값으로 바꾸고 나머지는 기준 200건의 값 그대로 두었을 때, 모델 예측의 평균."""
    mixed = crowd.copy()
    for col in known:
        mixed[col] = one[col].iloc[0]
    return float(model(mixed).mean())

worths = {known: worth(known) for r in range(7) for known in itertools.combinations(COLS, r)}       # 2의 6제곱 = 64가지
share = {}
for col in COLS:
    others = [c for c in COLS if c != col]
    total = 0.0
    for r in range(6):
        weight = math.factorial(r) * math.factorial(5 - r) / math.factorial(6)
        for known in itertools.combinations(others, r):
            with_col = tuple(c for c in COLS if c in known or c == col)
            total += weight * (worths[with_col] - worths[known])                 # 이 특성을 알려 주었을 때 예측이 변한 양
    share[col] = total
print({k: round(val, 3) for k, val in share.items()})
print(round(sum(share.values()), 3), round(float(model(one)[0]) - worths[()], 3), round(float(np.exp(worths[()])), 2), round(float(np.exp(model(one)[0])), 2))
alone_ = {col: round(worths[(col,)] - worths[()], 3) for col in COLS}            # 이 특성만 알려 주었을 때
last_ = {col: round(worths[tuple(COLS)] - worths[tuple(c for c in COLS if c != col)], 3) for col in COLS}   # 이 특성을 마지막에 알려 주었을 때
print(alone_)
print(last_)
print(round(float(crowd["달"].mean()), 1), round(float(crowd["면적"].mean()), 1))

In [ ]:
# 그림 14-8
order = sorted(share, key=lambda k: -abs(share[k]))
fig, ax = plt.subplots(figsize=(7.2, 3.9))
at = worths[()]
ax.axvline(np.exp(at), color=GRAY, lw=0.9, ls=":")
for k, col in enumerate(order):
    nxt = at + share[col]
    color = BRICK if share[col] > 0 else NAVY
    ax.barh(k, np.exp(nxt) - np.exp(at), left=np.exp(at), color=color, height=0.6)
    ax.text(max(np.exp(at), np.exp(nxt)) + 0.12, k, f"{share[col]:+.3f}", fontsize=7.5, va="center", color=color, bbox=BOX, zorder=5)
    at = nxt
ax.axvline(np.exp(at), color=TEAL, lw=1.2)
ax.set_yticks(range(len(order)), [f"{c} = {one[c].iloc[0]}" if c in ("면적", "층", "건축년도", "달") else c for c in order], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("예측 가격(억 원). 회색 점선: 기준 200건의 평균 예측, 청록색 선: 이 거래의 예측", fontsize=8)
lo, hi = ax.get_xlim()
ax.set_xlim(lo - 0.3, hi + 1.2)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_14_8.png", dpi=300)
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(C=1e6, max_iter=20000).fit(X[fit_on], ytr[fit_on])                       # 11장의 모델. 특성 41개
GROUPS = {"업력 구간(7개)": range(0, 7), "시군(30개)": range(7, 37), "지번폐업": [37], "주변": [38], "주변폐업": [39], "좌표없음": [40]}
before41 = auc(ytr[tune_on], logit.predict_proba(X[tune_on])[:, 1])
rng = np.random.default_rng(0)
for name, cols in GROUPS.items():
    falls = []
    for _ in range(10):
        mixed = X[tune_on].copy()
        mixed[:, list(cols)] = mixed[rng.permutation(len(mixed))][:, list(cols)]                    # 가변수로 나눈 특성은 그 가변수들을 함께 섞는다
        falls.append(before41 - auc(ytr[tune_on], logit.predict_proba(mixed)[:, 1]))
    print(name, round(float(np.mean(falls)), 4), round(float(np.std(falls)), 4))
print(round(float(logit.coef_[0][40]), 3), round(float(np.exp(logit.coef_[0][40])), 2), int(X[fit_on][:, 40].sum()), int(X[tune_on][:, 40].sum()), int(Xt[:, 40].sum()))

In [ ]:
shops["좌표없음"] = shops["위도"].isna()
shops["지금"] = np.where(shops["폐업"].isna(), "영업 중", "폐업")
print(shops.groupby("지금")["좌표없음"].agg(["size", "sum", "mean"]).round(3).to_string())
closed = shops[shops["폐업"].notna()]
when = pd.cut(closed["폐업"].dt.year, [1900, 2005, 2010, 2015, 2020, 2027], labels=["~2005", "2006~10", "2011~15", "2016~20", "2021~"])
miss_by = closed.groupby(when, observed=True)["좌표없음"].mean().round(3)
print(miss_by.to_dict())
gone = train["위도"].isna().to_numpy()
rate = pd.DataFrame({"기준일": train["기준일"].dt.year, "좌표없음": gone, "폐업함": ytr}).groupby(["기준일", "좌표없음"])["폐업함"].agg(["size", "mean"]).round(3)
print(rate.to_string())

In [ ]:
# 그림 14-9
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.5))
ax.bar(["영업 중"] + [f"폐업\n{k}" for k in miss_by.index], [float(shops.loc[shops["폐업"].isna(), "좌표없음"].mean()) * 100] + (miss_by * 100).tolist(),
       color=[TEAL] + [NAVY] * len(miss_by), width=0.62)
ax.set_ylabel("좌표가 없는 가게의 비율(%)", fontsize=8)
frame(ax, "지금 파일에서 좌표가 없는 비율")
ax.tick_params(axis="x", labelsize=6.8)
years_ = [2011, 2014, 2017, 2020]
w = 0.36
bx.bar(np.arange(4) - w / 2, [rate.loc[(k, False), "mean"] * 100 for k in years_], w, color=NAVY, label="좌표 있음")
bx.bar(np.arange(4) + w / 2, [rate.loc[(k, True), "mean"] * 100 for k in years_], w, color=BRICK, label="좌표 없음")
for k, yy in enumerate(years_):
    bx.text(k + w / 2, rate.loc[(yy, True), "mean"] * 100 + 1.2, f"{int(rate.loc[(yy, True), 'size'])}곳", fontsize=7, ha="center", color=BRICK)
bx.set_xticks(range(4), [f"{k}년" for k in years_], fontsize=8)
bx.set_ylabel("그 뒤 3년 안에 폐업한 비율(%)", fontsize=8)
bx.set_ylim(0, 75)
bx.legend(frameon=False, fontsize=7.5, loc="upper left")
frame(bx, "기준일별 폐업 비율")
fig.tight_layout()
fig.savefig("fig_14_9.png", dpi=300)
plt.show()

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

honest = LogisticRegression(C=1e6, max_iter=20000).fit(X[fit_on][:, :40], ytr[fit_on])              # 좌표없음 특성을 뺐다
for name, m, cols in [("로지스틱, 좌표없음 포함", logit, slice(None)), ("로지스틱, 좌표없음 제외", honest, slice(0, 40))]:
    print(name, round(auc(ytr[tune_on], m.predict_proba(X[tune_on][:, cols])[:, 1]), 3), round(auc(y, m.predict_proba(Xt[:, cols])[:, 1]), 3),
          round(brier(y, m.predict_proba(Xt[:, cols])[:, 1]), 4))

def seven(df, hide):
    """13.3절의 특성 7개. hide가 참이면 좌표가 없는 줄의 위도·경도·주변·주변폐업을 그 시군의 중앙값으로 채워서 좌표가 없다는 것을 알 수 없게 한다."""
    d = df.copy()
    if hide:
        for col in ["위도", "경도", "주변", "주변폐업"]:
            d[col] = d[col].fillna(d.groupby("시군")[col].transform("median")).fillna(d[col].median())
    return np.column_stack([d["업력"], d["위도"].fillna(0), d["경도"].fillna(0), d["지번폐업"], d["주변"].fillna(-1), d["주변폐업"].fillna(-1),
                            pd.Index(SIGUN).get_indexer(d["시군"])])

for name, hide in [("부스팅, 좌표 없는 줄을 그대로", False), ("부스팅, 좌표 없는 줄을 시군 중앙값으로", True)]:
    S, St = seven(train, hide), seven(test, hide)
    b = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=33, max_depth=3, early_stopping=False, random_state=0).fit(S[fit_on], ytr[fit_on])
    print(name, round(auc(ytr[tune_on], b.predict_proba(S[tune_on])[:, 1]), 3), round(auc(y, b.predict_proba(St)[:, 1]), 3), round(brier(y, b.predict_proba(St)[:, 1]), 4))
print(round(brier(y, np.full(len(y), ytr[fit_on].mean())), 4))